### 1.1 Basic Inference using pipeline()

The most basic object in the 🤗 Transformers library is the pipeline() function. It connects a model with its necessary preprocessing and postprocessing steps, allowing us to directly input any text and get an intelligible answer:

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

In [ ]:
# Passing multiple sentences
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

The pipeline() function supports multiple modalities, allowing you to work with text, images, audio, and even multimodal tasks. In this course we’ll focus on text tasks, but it’s useful to understand the transformer architecture’s potential, so we’ll briefly outline it.

#### 🤖 Machine Learning Pipelines Overview

This notebook outlines different types of pipelines used in machine learning, categorized by data modality.

---

#### 📝 Text Pipelines

- **`text-generation`**  
  Generate text from a given prompt.

- **`text-classification`**  
  Classify text into predefined categories.

- **`summarization`**  
  Create a shorter version of text while preserving key information.

- **`translation`**  
  Translate text from one language to another.

- **`zero-shot-classification`**  
  Classify text without prior training on specific labels.

- **`feature-extraction`**  
  Extract vector representations (embeddings) of text.

---

#### 🖼️ Image Pipelines

- **`image-to-text`**  
  Generate textual descriptions of images.

- **`image-classification`**  
  Identify objects or labels within an image.

- **`object-detection`**  
  Locate and identify multiple objects in an image.

---

#### 🎧 Audio Pipelines

- **`automatic-speech-recognition`**  
  Convert spoken language into text.

- **`audio-classification`**  
  Classify audio signals into categories.

- **`text-to-speech`**  
  Convert text into spoken audio.

---

#### 🔀 Multimodal Pipelines

- **`image-text-to-text`**  
  Generate a text response based on both an image and a text prompt.

---

In [ ]:
# Zero Shot Classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

In [ ]:
# Text Generation
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

In [ ]:
# Mask Filling
# The idea of this task is to fill in the blanks in a given text:
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

In [ ]:
# Named Entity Recognition
#ner = pipeline("ner", grouped_entities=True)  # old
ner = pipeline("ner", aggregation_strategy="simple")
print(ner("Elon Musk founded SpaceX in California"))

In [ ]:
# Question Answering
# The question-answering pipeline answers questions using information from a given context

'''
# transformers < 5.0
question_answerer = pipeline("document-question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Purushotham, I work in Mindful Learning",
)

'''


# Use a text-generation pipeline with a strong instruction/chat model
qa_generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.7B-Instruct")

context = "Hugging Face is a technology company based in New York and Paris. They are famous for their Transformers library."
question = "Where is Hugging Face located?"

# Combine the context and question into a prompt
prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"

# Generate the answer
result = qa_generator(prompt, max_new_tokens=50)

print(result[0]['generated_text'])


In [ ]:
# Sumarization

'''
# transformers<5.0

summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""
)
'''


In [ ]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="facebook/bart-large-cnn"
)

text = """
 America has changed dramatically during recent years. Not only has the number of 
    graduates in traditional engineering disciplines such as mechanical, civil, 
    electrical, chemical, and aeronautical engineering declined, but in most of 
    the premier American universities engineering curricula now concentrate on 
    and encourage largely the study of engineering science. As a result, there 
    are declining offerings in engineering subjects dealing with infrastructure, 
    the environment, and related issues, and greater concentration on high 
    technology subjects, largely supporting increasingly complex scientific 
    developments. While the latter is important, it should not be at the expense 
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other 
    industrial countries in Europe and Asia, continue to encourage and advance 
    the teaching of engineering. Both China and India, respectively, graduate 
    six and eight times as many traditional engineers as does the United States. 
    Other industrial countries at minimum maintain their output, while America 
    suffers an increasingly serious decline in the number of engineering graduates 
    and a lack of well-educated engineers.
"""

result = generator(
    "Summarize: " + text,
    max_length=50
)

print(result[0]["generated_text"])


- facebook/bart-large-cnn → ❗ not a text-generation model
- It is a seq2seq summarization model
- So pairing it with "text-generation" gives poor or broken output

In [ ]:
'''
generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base"   # works even with text-generation in v5
)
prompt = f"""
Summarize the following text into clear bullet points:

{text}

Bullet point summary:
1.
"""

result = generator(
    prompt,
    max_length=200,
    do_sample=False
)

print(result[0]["generated_text"])

'''
# Instead of pipeline() -   call the model properly using seq2seq APIs:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = f"""
Summarize the following text into clear bullet points:

{text}

Bullet point summary:
1.
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

outputs = model.generate(
    inputs["input_ids"],
    max_length=150,
    num_beams=4,
    early_stopping=True
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

### 1.2 Using Specific Models

The previous examples used the default model for the task at hand, but you can also choose a particular model from the Hub to use in a pipeline for a specific task — say, text generation. 

Go to the Model Hub and click on the corresponding tag on the left to display only the supported models for that task. 

Let’s try the HuggingFaceTB/SmolLM2-360M model! 

Here’s how to load it in the same pipeline as before:

- Explore models hub here: https://huggingface.co/models
- Explore text models here: https://huggingface.co/models?pipeline_tag=text-generation
- Explore model specific page here: https://huggingface.co/HuggingFaceTB/SmolLM2-360M

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

##### 1.2.1 Translation Example

In [ ]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
translator("Ce cours est produit par Hugging Face.")

In [ ]:
!pip install sentencepiece

In [ ]:
# Updated for 5.4
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

text = "Ce cours est produit par Hugging Face."

inputs = tokenizer(text, return_tensors="pt", truncation=True)

outputs = model.generate(inputs["input_ids"], max_length=100)

translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(translation)

##### 1.2.2 Image Classification

In [ ]:
from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224",
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

##### 1.2.3 Automatic Speech Recognition

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)

In [ ]:
!pip install soundfile

In [ ]:
import requests
import soundfile as sf
import io
from transformers import pipeline

url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)

audio_bytes = requests.get(url).content
audio, samplerate = sf.read(io.BytesIO(audio_bytes))

result = transcriber(audio)
print(result)

###### See details for quantization